# 🏆 Amazon ML Challenge 2026 — Business Entity Resolution

## Setup Instructions (do this ONCE before running)

### 1. Dataset
- Go to **Add Data** (right panel) → **Upload** your `6ab10eb3b23ba_student_resource.zip`
- OR attach the Kaggle dataset if you uploaded it there
- The dataset path will be `/kaggle/input/<your-dataset-name>/`

### 2. Accelerator
- Right panel → **Accelerator** → Select **GPU T4 x2** (or P100)
- Internet: can be OFF (all code is self-contained)

### 3. Run All
- **Run All** → grab a coffee ☕ (~45–60 min)

In [ ]:
# ── Cell 1: Detect environment & locate dataset ──────────────────────────────
import os, glob, shutil, sys

KAGGLE = os.path.exists('/kaggle')
print('Running on Kaggle:', KAGGLE)

if KAGGLE:
    # Find the zip file uploaded as a dataset
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    print('Found zips:', zips)
    
    # Working directory
    WORK_DIR = '/kaggle/working'
    DATA_DIR = os.path.join(WORK_DIR, 'dataset')
    OUTPUT_DIR = os.path.join(WORK_DIR, 'output')
    CODE_DIR = os.path.join(WORK_DIR, 'code')
else:
    WORK_DIR = os.getcwd()
    DATA_DIR = 'dataset'
    OUTPUT_DIR = 'output'
    CODE_DIR = 'code'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Work dir: {WORK_DIR}')
print(f'Data dir: {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Cell 2: Extract dataset from zip ─────────────────────────────────────────
import zipfile

if KAGGLE:
    zip_path = zips[0]  # Use first found zip
    print(f'Extracting {zip_path} ...')
    
    EXTRACT_TMP = os.path.join(WORK_DIR, '_extract_tmp')
    os.makedirs(EXTRACT_TMP, exist_ok=True)
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        entries = [
            e for e in z.namelist()
            if 'student_resource/dataset/' in e
            and '__MACOSX' not in e
            and '.DS_Store' not in e
        ]
        print(f'Extracting {len(entries)} dataset entries...')
        for entry in entries:
            # Strip 'student_resource/dataset/' prefix
            rel = entry.replace('student_resource/dataset/', '', 1)
            if not rel:
                continue
            dest = os.path.join(DATA_DIR, rel)
            if entry.endswith('/'):
                os.makedirs(dest, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                with z.open(entry) as src, open(dest, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                print(f'  {rel} ({os.path.getsize(dest)/1024**2:.0f} MB)')

# List extracted files
for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        fp = os.path.join(root, f)
        print(f'{fp}  ({os.path.getsize(fp)/1024**2:.1f} MB)')

In [ ]:
# ── Cell 3: Clone repo (enhanced-pipeline branch) ────────────────────────────
import subprocess

REPO_URL = 'https://github.com/ROHITH-KUMAR-L/amazon-ml.git'
BRANCH   = 'improvements/enhanced-pipeline'
REPO_DIR = os.path.join(WORK_DIR, 'amazon-ml')

if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} @ {BRANCH} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
else:
    print('Repo already exists, pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)

# Set up Python path to find src modules
SRC_ROOT = os.path.join(REPO_DIR, 'student_resource', 'code', 'business_entity_resolution')
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f'Source root: {SRC_ROOT}')
print('Contents:', os.listdir(os.path.join(SRC_ROOT, 'src')))

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
# Kaggle already has: pandas, numpy, sklearn, scipy, lightgbm, xgboost
# Only catboost may need installing
print('Installing/upgrading packages...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
     'lightgbm>=4.0.0', 'catboost>=1.2.0', 'xgboost>=2.0.0', 'tqdm'],
    capture_output=False
)

import lightgbm, catboost, xgboost, sklearn, scipy
print(f'LightGBM: {lightgbm.__version__}')
print(f'CatBoost: {catboost.__version__}')
print(f'XGBoost:  {xgboost.__version__}')
print(f'Sklearn:  {sklearn.__version__}')

# Confirm GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} — {props.total_memory/1024**3:.1f} GB VRAM')

In [ ]:
# ── Cell 5: Run the full pipeline ────────────────────────────────────────────
import time
t_start = time.time()

# Import pipeline after path is set
from src.pipeline import run_pipeline

run_pipeline(
    data_dir       = DATA_DIR,
    output_dir     = OUTPUT_DIR,
    sample_train_size = 0,      # 0 = use ALL training data (Kaggle has 30GB RAM)
    use_gpu        = True,
    neg_sample_ratio = 10.0,
)

elapsed = time.time() - t_start
print(f'\nTotal elapsed: {elapsed/60:.1f} minutes')

In [ ]:
# ── Cell 6: Validate submission ───────────────────────────────────────────────
VALIDATOR = os.path.join(REPO_DIR, 'student_resource', 'utils', 'validate_submission.py')
TEST_DIR  = os.path.join(DATA_DIR, 'test')

result = subprocess.run(
    [
        sys.executable, VALIDATOR,
        '--matching',   os.path.join(OUTPUT_DIR, 'matching_results.tsv'),
        '--candidate',  os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv'),
        '--test-dir',   TEST_DIR,
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERRORS:\n', result.stderr)
else:
    print('\n✅ VALIDATION PASSED — Safe to submit!')

In [ ]:
# ── Cell 7: Preview outputs ───────────────────────────────────────────────────
import pandas as pd

matching = pd.read_csv(os.path.join(OUTPUT_DIR, 'matching_results.tsv'), sep='\t', keep_default_na=False)
candidates = pd.read_csv(os.path.join(OUTPUT_DIR, 'candidate_pairs.tsv'), sep='\t', keep_default_na=False)

print('=== matching_results.tsv ===')
print(f'Total rows: {len(matching):,}')
matched = matching['matched_entity_ids'].str.len() > 0
print(f'Entities with matches:    {matched.sum():,}')
print(f'Singletons (no match):    {(~matched).sum():,}')
print()
print(matching.head(10).to_string(index=False))

print('\n=== candidate_pairs.tsv ===')
print(f'Total rows: {len(candidates):,}')
print(candidates.head(5).to_string(index=False))

print(f'\n📁 Output files saved to: {OUTPUT_DIR}')
for f in os.listdir(OUTPUT_DIR):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(fp)/1024**2:.1f} MB)')

## 📥 Download Output Files

After the notebook completes:
1. Go to the **Output** tab (right panel) → find `output/`
2. Download `matching_results.tsv` → upload to Amazon ML Challenge portal
3. Download `candidate_pairs.tsv` → include in final submission zip

## 📦 Final Submission Zip Structure
```
<team_name>_submission.zip
├── output/
│   ├── matching_results.tsv
│   └── candidate_pairs.tsv
├── code/
│   └── business_entity_resolution/
│       ├── src/
│       ├── README.md
│       └── requirements.txt
└── Documentation_template.md
```